# TELECOM CUSTOMER CHURN PREDICTION - Version 2
### improving my v1 project with AutoML, a business threshold and SHAP

**Author:** Ragini Gupta
**Dataset:** ibm telco customer churn (kaggle)

## 1. Importing Libraries

In [1]:
import os 
import warnings
warnings.filterwarnings("ignore")
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split , cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, roc_auc_score ,classification_report, ConfusionMatrixDisplay

from flaml import AutoML
import shap

## 2. Loading the Data

In [4]:
df = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print(f"{df.shape[0]:,} customers, {df.shape[1]} columns")
df.head()

7,043 customers, 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Cleaning the Data

In [5]:
# totalcharges is stored as text because of 11 blank rows
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# these 11 customers have tenure 0 (just joined, not billed yet) so total charges = 0
# in v1 i dropped them, now i keep them
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# customerid is just an id, not useful
df = df.drop('customerID', axis=1)

# churn: yes = 1, no = 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("missing values left:", df.isnull().sum().sum())
print("churn rate:", round(df['Churn'].mean() * 100, 1), "%")


missing values left: 0
churn rate: 26.5 %


## 4. Creating New Features

In [6]:
addon_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
              'TechSupport', 'StreamingTV', 'StreamingMovies']

def add_new_features(data):
    # how many extra services the customer has
    data['num_addons'] = (data[addon_cols] == 'Yes').sum(axis=1)
    # 1 if paying by automatic bank transfer / credit card
    data['auto_pay'] = data['PaymentMethod'].str.contains('automatic').astype(int)
    # 1 if fiber optic customer without tech support
    data['fiber_no_support'] = ((data['InternetService'] == 'Fiber optic') & (data['TechSupport'] != 'Yes')).astype(int)
    return data

df = add_new_features(df)
df[['num_addons', 'auto_pay', 'fiber_no_support']].head()

,num_addons,auto_pay,fiber_no_support
0,1,0,0
1,2,0,0
2,2,0,0
3,3,1,0
4,0,0,1
